In [ ]:
from lsst.summit.utils import ConsDbClient

In [ ]:
import numpy as np
from astropy.table import Table, join
from astropy.time import Time

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
%matplotlib widget

from matplotlib.ticker import ScalarFormatter
from scipy.signal import savgol_filter

import pandas as pd

from lsst.meas.algorithms.installGaussianPsf import FwhmPerSigma

from tqdm.notebook import tqdm

import os
import pandas as pd
from matplotlib.cm import get_cmap

In [ ]:
df = pd.read_pickle("./BLOCK_T614_df.pkl")

In [ ]:
for col in df.columns:
    print(col)

Let's start doing some stability metric calculations. We need some convenience functions to produce the metrics for trends. 

In [ ]:
import numpy as np
import pandas as pd

def _kendall_tau(x: np.ndarray, y: np.ndarray) -> float:
    """Kendall tau-b (no p-value); O(n^2) but fine for ~60 points."""
    n = len(y)
    if n < 3:
        return np.nan

    C = D = 0
    ties_x = 0
    ties_y = 0

    for i in range(n - 1):
        for j in range(i + 1, n):
            dx = x[j] - x[i]
            dy = y[j] - y[i]
            if dx == 0 and dy == 0:
                continue
            if dx == 0:
                ties_x += 1
                continue
            if dy == 0:
                ties_y += 1
                continue
            s = np.sign(dx) * np.sign(dy)
            if s > 0:
                C += 1
            elif s < 0:
                D += 1

    denom = np.sqrt((C + D + ties_x) * (C + D + ties_y))
    return (C - D) / denom if denom else np.nan

def drop_end_spike_if_needed(y, z=8.0):
    """
    If the final jump is z times larger than the typical jump, drop the last point.
    """
    y = np.asarray(y, dtype=float)
    if len(y) < 6 or not np.all(np.isfinite(y[-3:])):
        return slice(None)

    dy = np.diff(y)
    mad = np.median(np.abs(dy - np.median(dy)))
    # MAD->sigma-ish; add small floor to avoid divide-by-zero
    scale = 1.4826 * mad + 1e-12

    if abs(dy[-1]) > z * scale:
        return slice(0, -1)  # drop last point
    return slice(None)

def compute_run_stability(
    df: pd.DataFrame,
    value_col: str,
    group_cols=("night", "test_index"),
    p_low: float = 5.0,
    p_high: float = 95.0,
    *,
    time_col: str = "obs_start_dt",
    min_n_trend: int = 25,
    min_duration_min: float = 25.0,
    # Add to function signature
    trim_start: int = 0,
    trim_end: int = 1,
    drop_end_spike: bool = True,
    end_spike_z: float = 8.0,
) -> pd.DataFrame:
    """
    Compute stability metrics per run for a single column, including trend metrics.

    Trend gating:
    - Trend fields are computed only if the run has >= min_n_trend samples AND
      duration >= min_duration_min minutes (based on time_col).
    """

    group_cols = list(group_cols)

    def _spearman_rho(x: np.ndarray, y: np.ndarray) -> float:
        # simple spearman without scipy: rank then pearson
        xr = pd.Series(x).rank(method="average").to_numpy()
        yr = pd.Series(y).rank(method="average").to_numpy()
        xm = xr - xr.mean()
        ym = yr - yr.mean()
        denom = np.sqrt((xm * xm).sum()) * np.sqrt((ym * ym).sum())
        if denom == 0:
            return np.nan
        return float((xm * ym).sum() / denom)

    def _theil_sen_slope_per_min(x_min: np.ndarray, y: np.ndarray) -> float:
        # median of pairwise slopes (robust); O(n^2) but n~50-60 is fine
        n = len(y)
        if n < 2:
            return np.nan
        slopes = []
        for i in range(n - 1):
            dx = x_min[i+1:] - x_min[i]
            dy = y[i+1:] - y[i]
            m = dx != 0
            if np.any(m):
                slopes.extend((dy[m] / dx[m]).tolist())
        return float(np.median(slopes)) if slopes else np.nan

    def _agg_one_group(g: pd.DataFrame) -> pd.Series:
        # ----- basic stats on value_col -----
        s = pd.to_numeric(g[value_col], errors="coerce").dropna()

        if s.empty:
            return pd.Series({
                "mean": np.nan, "std": np.nan, "median": np.nan,
                "p_low": np.nan, "p_high": np.nan, "p_range": np.nan, "n": 0,
                "duration_min": np.nan,
                "trend_slope_per_min": np.nan,
                "trend_delta": np.nan,
                "trend_norm": np.nan,
                "trend_tau": np.nan,
            })

        p_lo, p_hi = np.nanpercentile(s, [p_low, p_high])

        # ----- duration and trend metrics -----
        # Build aligned time/value arrays (drop rows with bad time OR bad value)
        t = pd.to_datetime(g[time_col], errors="coerce")
        y = pd.to_numeric(g[value_col], errors="coerce")
        ok = t.notna() & np.isfinite(y)

        t = t[ok]
        y = y[ok].to_numpy(dtype=float)

        if len(t) >= 2:
            t_sorted_idx = np.argsort(t.to_numpy())
            t = t.iloc[t_sorted_idx]
            y = y[t_sorted_idx]
            duration_min = float((t.iloc[-1] - t.iloc[0]).total_seconds() / 60.0)
        else:
            duration_min = np.nan

        # Optional fixed trimming
        if (trim_start or trim_end) and len(y) > (trim_start + trim_end + 4):
            t = t.iloc[trim_start:len(y)-trim_end]
            y = y[trim_start:len(y)-trim_end]
        
        # Optional adaptive "end spike" removal
        if drop_end_spike and len(y) >= 6:
            dy = np.diff(y)
            mad = np.median(np.abs(dy - np.median(dy)))
            scale = 1.4826 * mad + 1e-12
            if np.isfinite(dy[-1]) and abs(dy[-1]) > end_spike_z * scale:
                t = t.iloc[:-1]
                y = y[:-1]
        
        trend_slope = np.nan
        trend_delta = np.nan
        trend_norm = np.nan
        trend_tau = np.nan

        # Gate trend computation
        if (
            np.isfinite(duration_min)
            and duration_min >= min_duration_min
            and len(y) >= min_n_trend
        ):
            # x in minutes from run start
            x_min = (t - t.iloc[0]).dt.total_seconds().to_numpy(dtype=float) / 60.0

            trend_slope = _theil_sen_slope_per_min(x_min, y)
            # IMPORTANT: use ACTUAL run duration, not a fixed 30 minutes
            trend_delta = float(trend_slope * duration_min) if np.isfinite(trend_slope) else np.nan

            # Robust scale for normalization (use p95-p5 of the values)
            scale = float(p_hi - p_lo)
            # prevent blow-ups when scale ~ 0
            if not np.isfinite(scale) or scale <= 0:
                scale = np.nan

            trend_norm = float(abs(trend_delta) / scale) if np.isfinite(trend_delta) and np.isfinite(scale) else np.nan
            trend_tau = _spearman_rho(x_min, y)

        return pd.Series({
            "mean": float(s.mean()),
            "std": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
            "median": float(s.median()),
            "p_low": float(p_lo),
            "p_high": float(p_hi),
            "p_range": float(p_hi - p_lo),
            "n": int(len(s)),
            "duration_min": duration_min,
            "trend_slope_per_min": trend_slope,
            "trend_delta": trend_delta,
            "trend_norm": trend_norm,
            "trend_tau": trend_tau,
        })

    out = (
        df.groupby(group_cols, dropna=False)
          .apply(_agg_one_group)
          .reset_index()
    )

    # Rename metrics to include the column name
    rename_map = {
        "mean": f"{value_col}_mean",
        "std": f"{value_col}_std",
        "median": f"{value_col}_median",
        "p_low": f"{value_col}_p{int(p_low)}",
        "p_high": f"{value_col}_p{int(p_high)}",
        "p_range": f"{value_col}_p{int(p_low)}_p{int(p_high)}_range",
        "n": f"{value_col}_n",
        "duration_min": f"{value_col}_duration_min",
        "trend_slope_per_min": f"{value_col}_trend_slope_per_min",
        "trend_delta": f"{value_col}_trend_delta",
        "trend_norm": f"{value_col}_trend_norm",
        "trend_tau": f"{value_col}_trend_tau",
    }
    out = out.rename(columns=rename_map)

    return out


In [ ]:
def plot_test_timeseries(df, value_col, night, test_index):
    d = df[(df['night'] == night) & (df['test_index'] == test_index)].copy()
    d = d.sort_values('obs_start_dt')

    plt.figure(figsize=(12,4))
    plt.plot(d['obs_start_dt'], d[value_col], marker='o')
    plt.title(f"{value_col} during test {night.date()} T{test_index}")
    plt.xlabel("Time")
    plt.ylabel(value_col)
    plt.ticklabel_format(axis="y", style="plain", useOffset=False)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
#plot_test_timeseries(df, 'camera_hexapod_compensation_offset_x', night=pd.Timestamp('2025-11-07'), test_index=1)

In [ ]:
stab_fwhm = compute_run_stability(df, 'psf_fwhm_50')
stab_aos_fwhm = compute_run_stability(df, 'aos_fwhm_2')
stab_e50  = compute_run_stability(df, 'psf_e_50')
stab_fwhm_95_0  = compute_run_stability(df, 'psf_fwhm_95_05')

In [ ]:
stab_aos_fwhm

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

sc = ax.scatter(
    np.abs(stab_aos_fwhm["aos_fwhm_2_trend_tau"]),
    np.abs(stab_aos_fwhm["aos_fwhm_2_trend_norm"]),
    s=60,
    picker=True,
)

ax.set_xlabel("AOS FWHM |Trend Tau|")
ax.set_ylabel("AOS FWHM |Trend Norm|")
ax.set_title("Stability: AOS FWHM Trend Tau vs Trend Norm")
ax.grid(True)

annot = ax.annotate(
    "",
    xy=(0, 0),
    xytext=(10, 10),
    textcoords="offset points",
    bbox=dict(boxstyle="round", fc="w"),
    arrowprops=dict(arrowstyle="->"),
)
annot.set_visible(False)

def on_pick(event):
    idx = event.ind[0]
    row = stab_aos_fwhm.iloc[idx]

    annot.xy = (
        np.abs(row["aos_fwhm_2_trend_tau"]),
        np.abs(row["aos_fwhm_2_trend_norm"]),
    )
    annot.set_text(f"night={row['night']}\ntest_index={row['test_index']}")
    annot.set_visible(True)
    fig.canvas.draw_idle()

fig.canvas.mpl_connect("pick_event", on_pick)

plt.show()

In [ ]:
# Make a matching test_id (same pattern you used in test_stats)
stab_aos_fwhm['night_str'] = stab_aos_fwhm['night'].dt.strftime('%Y-%m-%d')
stab_aos_fwhm['test_id'] = (
    stab_aos_fwhm['night_str'] + '_T' + stab_aos_fwhm['test_index'].astype(int).astype(str)
)

In [ ]:
stress_by_test = (
    df.groupby(["night", "test_index"])[
        ["mtaos_stressM1M3", "mtaos_stressM2"]
    ]
    .median()   # or .first()
    .reset_index()
)

In [ ]:
stab_with_stress = stab_aos_fwhm.merge(
    stress_by_test,
    on=["night", "test_index"],
    how="left",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))

x = np.abs(stab_with_stress["aos_fwhm_2_trend_tau"])
y1 = np.abs(stab_with_stress["mtaos_stressM1M3"])
y2 = np.abs(stab_with_stress["mtaos_stressM2"])

sc1 = ax.scatter(x, y1, s=70, label="M1M3 Stress", picker=True)
sc2 = ax.scatter(x, y2, s=70, marker="s", label="M2 Stress", picker=True)

ax.set_xlabel("AOS FWHM |Trend Tau|")
ax.set_ylabel("|Mirror Stress|")
ax.set_title("AOS FWHM Trend Tau vs Mirror Stresses")
ax.legend()
ax.grid(True)

annot = ax.annotate(
    "",
    xy=(0, 0),
    xytext=(10, 10),
    textcoords="offset points",
    bbox=dict(boxstyle="round", fc="w"),
    arrowprops=dict(arrowstyle="->"),
)
annot.set_visible(False)

def on_pick(event):
    idx = event.ind[0]
    row = stab_with_stress.iloc[idx]

    annot.xy = (
        np.abs(row["aos_fwhm_2_trend_tau"]),
        np.abs(
            row["mtaos_stressM1M3"]
            if event.artist is sc1
            else row["mtaos_stressM2"]
        ),
    )

    annot.set_text(
        f"night={row['night']}\n"
        f"test_index={row['test_index']}"
    )
    annot.set_visible(True)
    fig.canvas.draw_idle()

fig.canvas.mpl_connect("pick_event", on_pick)

plt.show()

In [ ]:
def plot_test_mosaic(df, value_col: str, max_tests: int | None = None):
    """
    Make a mosaic of time series for each test execution.

    Parameters
    ----------
    df : DataFrame
        Must have 'night', 'test_index', 't_since_start_min', and value_col.
    value_col : str
        Column to plot (e.g. 'psf_fwhm_50').
    max_tests : int or None
        If set, limit to the first N tests (after sorting) to avoid huge mosaics.
    """
    # Unique tests
    tests = (
        df[['night', 'test_index']]
        .drop_duplicates()
        .sort_values(['night', 'test_index'])
        .reset_index(drop=True)
    )

    if max_tests is not None:
        tests = tests.head(max_tests)

    n = len(tests)
    if n == 0:
        print("No tests found.")
        return

    fig, axes = plt.subplots(n, 1, figsize=(12, 3*n), sharex=True)

    # If only one test, axes is not an array
    if n == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, tests.iterrows()):
        mask = ((df['night'] == row['night']) &
                (df['test_index'] == row['test_index']))
        d = df.loc[mask, ['t_since_start_min', value_col]].copy()

        # Drop NaNs in x or y
        d = d.dropna(subset=['t_since_start_min', value_col])

        if d.empty:
            ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                    ha='center', va='center')
        else:
            d = d.sort_values('t_since_start_min')
            ax.plot(d['t_since_start_min'], d[value_col], marker='o')
        
        night_str = row['night'].strftime('%Y-%m-%d')
        ax.set_title(f"{night_str}  T{int(row['test_index'])}")
        ax.set_ylabel(value_col)
        ax.grid(True)

    axes[-1].set_xlabel("Minutes since test start")
    plt.tight_layout()
    plt.show()

In [ ]:
def add_test_relative_time(
    df: pd.DataFrame,
    time_col: str = "obs_start_dt",
    group_cols = ("night", "test_index"),
    out_col: str = "t_since_start_min",
) -> pd.DataFrame:
    """Add minutes since test start for each (night, test_index) group."""
    df = df.copy()
    group_cols = list(group_cols)

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce", utc=True)
    starts = df.groupby(group_cols)[time_col].transform("min")
    dt = (df[time_col] - starts).dt.total_seconds() / 60.0
    df[out_col] = dt
    return df


def plot_spaghetti_normalized(
    df: pd.DataFrame,
    value_col: str,
    max_tests: int | None = None,
    first_n: int = 5,
):
    """
    Spaghetti plot: all tests overplotted, normalized by the mean
    of the first N datapoints in each test. Click a line to highlight it.

    Parameters
    ----------
    df : DataFrame
        Must have 'night', 'test_index', 'obs_start_dt', and value_col.
    value_col : str
        Column to plot, e.g. 'psf_fwhm_50'.
    max_tests : int or None
        If None (default), plot ALL tests. Otherwise limit to first N.
    first_n : int
        Number of initial points to use for normalization.
    """
    # Ensure relative time exists
    if "t_since_start_min" not in df.columns:
        df_local = add_test_relative_time(df)
    else:
        df_local = df.copy()

    # Coerce value to numeric
    df_local[value_col] = pd.to_numeric(df_local[value_col], errors="coerce")

    # Unique tests
    tests = (
        df_local[["night", "test_index"]]
        .drop_duplicates()
        .sort_values(["night", "test_index"])
        .reset_index(drop=True)
    )
    if max_tests is not None:
        tests = tests.head(max_tests)

    n_tests = len(tests)
    if n_tests == 0:
        print("No tests found.")
        return

    fig, ax = plt.subplots(figsize=(10, 6))

    lines = []
    labels = []
    base_colors = []

    # Use a categorical colormap for distinct test colors
    cmap = get_cmap("tab20")
    # Repeat colors if more tests than colors in the map
    n_colors = cmap.N

    for idx, (_, row) in enumerate(tests.iterrows()):
        mask = ((df_local["night"] == row["night"]) &
                (df_local["test_index"] == row["test_index"]))
        d = df_local.loc[mask, ["t_since_start_min", value_col]].dropna()
        if len(d) < 2:
            continue

        d = d.sort_values("t_since_start_min")

        # Normalization factor = mean of first_n points
        base = d[value_col].iloc[:first_n].mean()
        if not np.isfinite(base) or base == 0:
            continue

        y_norm = d[value_col] / base
        x_rel = d["t_since_start_min"]

        night_str = pd.to_datetime(row["night"]).strftime("%Y-%m-%d")
        test_id = f"{night_str}_T{int(row['test_index'])}"

        color = cmap(idx % n_colors)
        line, = ax.plot(
            x_rel,
            y_norm,
            color=color,
            linewidth=1.5,
            alpha=0.7,
            picker=5,   # pick radius
        )
        lines.append(line)
        labels.append(test_id)
        base_colors.append(color)

    ax.set_xlabel("Minutes since test start")
    ax.set_ylabel(f"{value_col} / mean(first {first_n})")
    ax.set_title(
        f"{value_col}: normalized spaghetti plot "
        f"({len(lines)} tests; click to highlight)"
    )
    ax.grid(True)

    # --- Click-to-highlight logic -----------------------------------------
    def on_pick(event):
        thisline = event.artist
        if thisline not in lines:
            return

        # Reset all lines to faint but keep their colors
        for ln, col in zip(lines, base_colors):
            ln.set_linewidth(1.0)
            ln.set_alpha(0.2)
            ln.set_color(col)
            ln.set_zorder(1)

        # Highlight selected line
        idx = lines.index(thisline)
        thisline.set_linewidth(3.0)
        thisline.set_alpha(1.0)
        thisline.set_zorder(10)

        test_label = labels[idx]
        ax.set_title(
            f"{value_col}: normalized spaghetti plot — highlighted {test_label}"
        )
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("pick_event", on_pick)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_spaghetti_normalized(df, "aos_fwhm_2")

In [ ]:
#plot_spaghetti_normalized(df, "psf_fwhm_95_05")

In [ ]:
def plot_test_timeseries_panels(
    df,
    value_cols,
    night,
    test_index,
    title=None,
    ylabels=None,
    figsize=None,
    time_col="obs_start_dt",
    height_per_panel=2.5,   # tweak for tall figures
    max_panels=None,        # e.g. 12 to prevent accidental 80-panel monsters
):
    """
    Plot N time-series columns as stacked panels for a single (night, test_index),
    sharing the same time axis.

    Parameters
    ----------
    df : DataFrame
        Must include 'night', 'test_index', and time_col.
    value_cols : list[str] or str
        1..N column names to plot.
    night : int, date, or Timestamp
        Night identifier. If int, interpreted as YYYYMMDD.
    test_index : int
        Test execution index.
    title : str or None
        Overall figure title. If None, auto-generated.
    ylabels : list[str] or None
        Per-panel y-axis labels. If None, defaults to value_cols.
    figsize : tuple or None
        Matplotlib figure size. If None, chosen based on #panels.
    time_col : str
        Name of the time column (e.g. 'obs_start_dt').
    height_per_panel : float
        Figure height (inches) per panel when figsize is None.
    max_panels : int or None
        Optional safety limit on number of panels.
    """

    # Normalize value_cols to list
    if isinstance(value_cols, str):
        value_cols = [value_cols]
    if len(value_cols) < 1:
        raise ValueError("value_cols must contain at least 1 column.")
    if max_panels is not None and len(value_cols) > max_panels:
        raise ValueError(f"value_cols has {len(value_cols)} columns; max_panels={max_panels}")

    # Handle night passed as int (YYYYMMDD) if df['night'] is datetime-like
    night_series = df["night"]
    if isinstance(night, int):
        if pd.api.types.is_datetime64_any_dtype(night_series):
            night_dt = pd.to_datetime(str(night), format="%Y%m%d")
            night_key = night_dt.normalize()
        else:
            night_key = night
    else:
        night_key = night

    # Filter to this test
    d = df[(df["night"] == night_key) & (df["test_index"] == test_index)].copy()
    if d.empty:
        print(f"No rows found for night={night_key}, test_index={test_index}")
        return

    # Clean and sort time
    d = d.dropna(subset=[time_col])
    d[time_col] = pd.to_datetime(d[time_col], errors="coerce", utc=True)
    d = d.dropna(subset=[time_col]).sort_values(time_col)

    if d.empty:
        print("No valid timestamps after cleaning; nothing to plot.")
        return

    # Default ylabels
    if ylabels is None:
        ylabels = value_cols
    else:
        if len(ylabels) != len(value_cols):
            raise ValueError("ylabels must have the same length as value_cols.")

    n_panels = len(value_cols)
    if figsize is None:
        figsize = (12, max(3, height_per_panel * n_panels))

    fig, axes = plt.subplots(n_panels, 1, figsize=figsize, sharex=True)
    # Ensure axes is always an indexable list-like
    if n_panels == 1:
        axes = [axes]

    for ax, col, ylabel in zip(axes, value_cols, ylabels):
        if col not in d.columns:
            ax.text(0.5, 0.5, f"{col} missing", transform=ax.transAxes,
                    ha="center", va="center")
            ax.set_ylabel(ylabel)
            ax.grid(True)
            continue

        dd = d.dropna(subset=[col])
        if dd.empty:
            ax.text(0.5, 0.5, f"No data for {col}", transform=ax.transAxes,
                    ha="center", va="center")
            ax.set_ylabel(ylabel)
            ax.grid(True)
            continue

        ax.plot(dd[time_col], dd[col], marker="o")
        ax.set_ylabel(ylabel)
        ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        ax.grid(True)

    axes[-1].set_xlabel("Time")

    if title is None:
        night_str = night_key.date().isoformat() if isinstance(night_key, pd.Timestamp) else str(night_key)
        cols_str = ", ".join(value_cols)
        title = f"{cols_str} during test {night_str} T{test_index}"

    fig.suptitle(title, y=0.98)
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()


In [ ]:
plot_test_timeseries_panels(
    df,
    value_cols=["aos_fwhm_2", "z4_2", "z_gradient", "dimm_seeing"],
    night=pd.Timestamp('2025-10-26'),
    test_index=0,
    ylabels=["AOS FWHM", "Z4", "z-gradient", "DIMM Seeing"],
    title="BLOCK-T614 2025-10-26 1st Execution"
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from matplotlib.colors import Normalize
from matplotlib.ticker import ScalarFormatter

def plot_spaghetti_centered_by_aos_tau(
    df: pd.DataFrame,
    value_col: str,
    stab_aos_fwhm: pd.DataFrame,
    *,
    aos_tau_col: str = "aos_fwhm_2_trend_tau",
    max_tests: int | None = None,
    first_n: int = 5,
    abs_tau: bool = True,
    show_colorbar: bool=True,
    ylabel: str | None = None,      # NEW
    title: str | None = None,       # NEW
):
    """
    Spaghetti plot of value_col vs time, centered by subtracting the mean of the
    first N points in each run. Line color encodes AOS FWHM trend_tau.

    No click interaction; purely visual.

    Parameters
    ----------
    df : DataFrame
        Must have 'night', 'test_index', and 'obs_start_dt' (or t_since_start_min).
    value_col : str
        Column to plot.
    stab_aos_fwhm : DataFrame
        Per-run stability table containing aos_tau_col.
    aos_tau_col : str
        Column name with the AOS FWHM trend tau metric.
    max_tests : int or None
        Limit number of runs plotted (None = all).
    first_n : int
        Number of initial points used to compute baseline subtraction.
    abs_tau : bool
        If True, color by |tau| so strong monotonic trends stand out regardless of sign.
    """
    # --- ensure relative time exists ---
    if "t_since_start_min" not in df.columns:
        df_local = add_test_relative_time(df)
    else:
        df_local = df.copy()

    # --- numeric coercion ---
    df_local[value_col] = pd.to_numeric(df_local[value_col], errors="coerce")

    # --- merge tau per run ---
    keys = ["night", "test_index"]
    tau_df = stab_aos_fwhm[keys + [aos_tau_col]].copy()
    tau_df[aos_tau_col] = pd.to_numeric(tau_df[aos_tau_col], errors="coerce")

    df_local = df_local.merge(tau_df, on=keys, how="left")

    # --- select runs ---
    tests = (
        df_local[keys]
        .drop_duplicates()
        .sort_values(keys)
        .reset_index(drop=True)
    )
    if max_tests is not None:
        tests = tests.head(max_tests)

    if len(tests) == 0:
        print("No tests found.")
        return

    # --- color mapping (viridis) ---
    tau_vals = pd.to_numeric(tau_df[aos_tau_col], errors="coerce")
    tau_vals = tau_vals[np.isfinite(tau_vals)]
    if abs_tau:
        tau_vals = np.abs(tau_vals)

    if tau_vals.size == 0:
        vmin, vmax = 0.0, 1.0
    else:
        vmin, vmax = float(np.nanmin(tau_vals)), float(np.nanmax(tau_vals))
        if not np.isfinite(vmax) or vmax <= vmin:
            vmax = vmin + 1e-6

    norm = Normalize(vmin=vmin, vmax=vmax, clip=True)
    cmap = get_cmap("viridis")

    fig, ax = plt.subplots(figsize=(10, 6))

    n_plotted = 0

    for _, row in tests.iterrows():
        mask = (
            (df_local["night"] == row["night"]) &
            (df_local["test_index"] == row["test_index"])
        )
        d = df_local.loc[mask, ["t_since_start_min", value_col, aos_tau_col]]
        d = d.dropna(subset=["t_since_start_min", value_col])

        if len(d) < 2:
            continue

        d = d.sort_values("t_since_start_min")

        base = d[value_col].iloc[:first_n].mean()
        if not np.isfinite(base):
            continue

        # Center by subtraction (preserves units)
        y_centered = d[value_col] - base
        x_rel = d["t_since_start_min"].to_numpy()

        tau_run = pd.to_numeric(d[aos_tau_col], errors="coerce").iloc[0]
        tau_color_val = abs(tau_run) if abs_tau and np.isfinite(tau_run) else tau_run

        color = (
            cmap(norm(tau_color_val))
            if np.isfinite(tau_color_val)
            else (0.6, 0.6, 0.6, 0.4)
        )

        ax.plot(
            x_rel,
            y_centered,
            color=color,
            linewidth=1.2,
            alpha=0.75,
        )
        n_plotted += 1

    ax.axhline(0.0, linestyle="--", linewidth=1.2)
    ax.set_xlabel("Minutes since test start")
    ax.set_ylabel(f"{value_col} − mean(first {first_n})")
    ax.set_title(
        f"{value_col}: centered spaghetti plot "
        f"({n_plotted} runs; color = {'|tau|' if abs_tau else 'tau'} from {aos_tau_col})"
    )
    ax.grid(True)

    # Disable scientific notation on y-axis
    fmt = ScalarFormatter(useMathText=False)
    fmt.set_scientific(False)
    fmt.set_useOffset(False)
    ax.yaxis.set_major_formatter(fmt)

    if show_colorbar:
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02)
        label = f"|{aos_tau_col}|" if abs_tau else aos_tau_col
        cbar.set_label(label)

    # ✅ Optional labels
    ax.set_ylabel(ylabel if ylabel is not None else value_col)

    if title is not None:
        ax.set_title(title)
    else:
        ax.set_title(f"{value_col} centered by first {first_n} points")

    
    plt.tight_layout()
    plt.show()

In [ ]:
for col in df.columns:
    print(col)

In [ ]:
df["m1m3_ims_xPosition"] = df["m1m3_ims_xPosition"]*1000000.

In [ ]:
df["m1m3_ims_yPosition"] = df["m1m3_ims_yPosition"]*1000000.

In [ ]:
df["m1m3_ims_zPosition"] = df["m1m3_ims_zPosition"]*1000000.

In [ ]:
# stab_aos_fwhm should already contain aos_fwhm_2_trend_tau per (night,test_index)
plot_spaghetti_centered_by_aos_tau(
    df,
    value_col="mtcamera_domeYMinusTemp",
    stab_aos_fwhm=stab_aos_fwhm,
    first_n=5,
    max_tests=None,
    abs_tau=True,   # color by |tau| so big monotonic trends stand out
)

In [ ]:
# stab_aos_fwhm should already contain aos_fwhm_2_trend_tau per (night,test_index)
plot_spaghetti_centered_by_aos_tau(
    df,
    value_col="aos_fwhm_2",
    stab_aos_fwhm=stab_aos_fwhm,
    first_n=5,
    max_tests=None,
    abs_tau=True,   # color by |tau| so big monotonic trends stand out
    ylabel="$\Delta$ AOS FWHM",
    title="Change in AOS FWHM vs AOS FWHM Trend"
)

In [ ]:
# stab_aos_fwhm should already contain aos_fwhm_2_trend_tau per (night,test_index)
plot_spaghetti_centered_by_aos_tau(
    df,
    value_col="tma_truss_xy",
    stab_aos_fwhm=stab_aos_fwhm,
    first_n=5,
    max_tests=None,
    abs_tau=True,   # color by |tau| so big monotonic trends stand out
)

In [ ]:
# stab_aos_fwhm should already contain aos_fwhm_2_trend_tau per (night,test_index)
plot_spaghetti_centered_by_aos_tau(
    df,
    value_col="m1m3_ims_zPosition",
    stab_aos_fwhm=stab_aos_fwhm,
    first_n=5,
    max_tests=None,
    abs_tau=True,   # color by |tau| so big monotonic trends stand out
    ylabel="$\Delta$ M1M3 IMS Z position (microns)",
    title="Change in M1M3 Z Position vs AOS FWHM Trend"
)

In [ ]:
# stab_aos_fwhm should already contain aos_fwhm_2_trend_tau per (night,test_index)
plot_spaghetti_centered_by_aos_tau(
    df,
    value_col="m1m3_ims_xPosition",
    stab_aos_fwhm=stab_aos_fwhm,
    first_n=5,
    max_tests=None,
    abs_tau=True,   # color by |tau| so big monotonic trends stand out
    ylabel="$\Delta$ M1M3 IMS X position (microns)",
    title="Change in M1M3 X Position vs AOS FWHM Trend"
)

In [ ]:
# stab_aos_fwhm should already contain aos_fwhm_2_trend_tau per (night,test_index)
plot_spaghetti_centered_by_aos_tau(
    df,
    value_col="m1m3_ims_yPosition",
    stab_aos_fwhm=stab_aos_fwhm,
    first_n=5,
    max_tests=None,
    abs_tau=True,   # color by |tau| so big monotonic trends stand out
    ylabel="$\Delta$ M1M3 IMS Y position (microns)",
    title="Change in M1M3 Y Position vs AOS FWHM Trend"
)

ChatGPT analysis

In [ ]:
aos_key_tau  = "aos_fwhm_2_trend_tau"
aos_key_norm = "aos_fwhm_2_trend_norm"

interesting_runs = stab_aos_fwhm[
    (stab_aos_fwhm[aos_key_tau].abs() > 0.5) &
    (stab_aos_fwhm[aos_key_norm].abs() > 0.5)
][["night", "test_index"]]

Trying a new approach, rule things out. 

In [ ]:
def run_trend_compare(df, y_col, x_col, group_cols=("night","test_index"), time_col="obs_start_dt",
                      min_duration_min=25, min_n_trend=25):
    """
    Returns per-run table with:
      - keys
      - y trend_norm / trend_tau
      - x trend_norm / trend_tau
    """
    y = compute_run_stability(
        df, y_col, group_cols=group_cols, time_col=time_col,
        min_duration_min=min_duration_min, min_n_trend=min_n_trend
    )[[*group_cols, f"{y_col}_trend_norm", f"{y_col}_trend_tau", f"{y_col}_duration_min"]]

    x = compute_run_stability(
        df, x_col, group_cols=group_cols, time_col=time_col,
        min_duration_min=min_duration_min, min_n_trend=min_n_trend
    )[[*group_cols, f"{x_col}_trend_norm", f"{x_col}_trend_tau"]]

    out = y.merge(x, on=list(group_cols), how="inner")
    return out

In [ ]:
def no_sci(ax):
    fmt = ScalarFormatter(useMathText=False)
    fmt.set_scientific(False)
    fmt.set_useOffset(False)
    ax.yaxis.set_major_formatter(fmt)

def dense_time_ticks(ax, minticks=8, maxticks=14, rotate=0):
    locator = mdates.AutoDateLocator(minticks=minticks, maxticks=maxticks)
    formatter = mdates.ConciseDateFormatter(locator)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    if rotate:
        for lab in ax.get_xticklabels():
            lab.set_rotation(rotate)
            lab.set_ha("right")

# ---------- choose the comparison ----------
Y_COL = "aos_fwhm_2"
X_COL = "m2_hexapod_compensation_offset_z"   # <-- swap this to any candidate column you want

GROUP_COLS = ("night","test_index")
TIME_COL = "obs_start_dt"

cmp = run_trend_compare(
    df, Y_COL, X_COL,
    group_cols=GROUP_COLS,
    time_col=TIME_COL,
    min_duration_min=25,
    min_n_trend=25,
)

x_norm_col = f"{X_COL}_trend_norm"
y_norm_col = f"{Y_COL}_trend_norm"

# Use absolute values if you only care about magnitude; keep signed if direction matters
plot_df = cmp.copy()
x = np.abs(pd.to_numeric(plot_df[x_norm_col], errors="coerce").to_numpy())
y = np.abs(pd.to_numeric(plot_df[y_norm_col], errors="coerce").to_numpy())

mask = np.isfinite(x) & np.isfinite(y)
plot_df = plot_df.loc[mask].reset_index(drop=True)
x = x[mask]
y = y[mask]

# ---------- figure layout ----------
fig = plt.figure(figsize=(10, 8), constrained_layout=True)
gs = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[2.2, 1.4, 1.4])

ax_sc = fig.add_subplot(gs[0, 0])
ax_y  = fig.add_subplot(gs[1, 0])
ax_x  = fig.add_subplot(gs[2, 0], sharex=ax_y)

sc = ax_sc.scatter(x, y, alpha=0.7, picker=True, pickradius=5)
ax_sc.set_xlabel(f"|{x_norm_col}|")
ax_sc.set_ylabel(f"|{y_norm_col}|")
ax_sc.set_title(f"Run instability comparison: {Y_COL} vs {X_COL} (click a run)")
ax_sc.grid(True)
no_sci(ax_sc)

annot = ax_sc.annotate(
    "",
    xy=(0, 0),
    xytext=(10, 10),
    textcoords="offset points",
    bbox=dict(boxstyle="round", fc="w"),
    arrowprops=dict(arrowstyle="->"),
)
annot.set_visible(False)

# init bottom panels
ax_y.set_title("Click a point above to load run time series")
ax_y.set_ylabel(Y_COL)
ax_y.grid(True)
no_sci(ax_y)

ax_x.set_ylabel(X_COL)
ax_x.set_xlabel("Time")
ax_x.grid(True)
no_sci(ax_x)
dense_time_ticks(ax_x)

def _load_run_ts(night, test_index):
    d = df[(df["night"] == night) & (df["test_index"] == test_index)].copy()
    if d.empty:
        return d
    d[TIME_COL] = pd.to_datetime(d[TIME_COL], errors="coerce")
    d[Y_COL] = pd.to_numeric(d[Y_COL], errors="coerce")
    d[X_COL] = pd.to_numeric(d[X_COL], errors="coerce")
    return d.dropna(subset=[TIME_COL]).sort_values(TIME_COL)

def on_pick(event):
    if event.artist is not sc:
        return

    idx = int(event.ind[0])
    row = plot_df.iloc[idx]
    night = row["night"]
    test_index = row["test_index"]

    annot.xy = (x[idx], y[idx])
    annot.set_text(f"night={night}\ntest_index={test_index}")
    annot.set_visible(True)

    d = _load_run_ts(night, test_index)

    ax_y.cla()
    ax_x.cla()
    ax_y.grid(True)
    ax_x.grid(True)
    ax_x.set_xlabel("Time")

    if d.empty:
        ax_y.set_title(f"Run night={night}, test_index={test_index} (no data)")
        fig.canvas.draw_idle()
        return

    ax_y.plot(d[TIME_COL], d[Y_COL], marker="o", markersize=3, linewidth=1.2)
    ax_y.set_ylabel(Y_COL)
    no_sci(ax_y)

    ax_x.plot(d[TIME_COL], d[X_COL], marker="s", markersize=3, linewidth=1.2)
    ax_x.set_ylabel(X_COL)
    no_sci(ax_x)

    plt.setp(ax_y.get_xticklabels(), visible=False)
    dense_time_ticks(ax_x, minticks=10, maxticks=18)

    ax_y.set_title(
        f"Run night={night}, test_index={test_index} | "
        f"|{x_norm_col}|={abs(row[x_norm_col]):.3g}, |{y_norm_col}|={abs(row[y_norm_col]):.3g}"
    )

    fig.canvas.draw_idle()

fig.canvas.mpl_connect("pick_event", on_pick)

plt.show()


In [ ]:
import numpy as np
import pandas as pd

def compute_composite_thermal_activity_from_raw(
    df: pd.DataFrame,
    temp_cols: list[str],
    *,
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
    first_n: int = 5,
    p_low: float = 5.0,
    p_high: float = 95.0,
    robust: bool = True,
) -> pd.DataFrame:
    """
    Compute a composite per-run thermal activity metric directly from raw
    time-series data, including computing trend_norm for each temperature column.

    Parameters
    ----------
    df : DataFrame
        Raw time-series dataframe containing temperature columns.
    temp_cols : list[str]
        Temperature column names to include.
    group_cols : tuple[str]
        Run identifier columns.
    time_col : str
        Time column used for trend fitting.
    first_n : int
        Number of initial points used to define baseline scale for trend_norm.
    p_low, p_high : float
        Percentiles used for robust scale estimation.
    robust : bool
        If True, normalize each temperature topic by its median trend across runs.

    Returns
    -------
    DataFrame
        Columns:
        - group_cols
        - thermal_activity
        - thermal_n_used
        - plus individual `{col}_trend_norm` columns
    """
    group_cols = list(group_cols)

    # --- compute trend_norm per temperature column ---
    stab_tables = []

    for col in temp_cols:
        stab = compute_run_stability(
            df,
            value_col=col,
            group_cols=group_cols,
            p_low=p_low,
            p_high=p_high,
        )
        stab_tables.append(stab)

    # --- merge all stability tables ---
    stab_all = stab_tables[0]
    for stab in stab_tables[1:]:
        stab_all = stab_all.merge(stab, on=group_cols, how="outer")

    # --- extract trend_norm columns ---
    trend_cols = [f"{c}_trend_norm" for c in temp_cols]
    missing = [c for c in trend_cols if c not in stab_all.columns]
    if missing:
        raise RuntimeError(f"Missing trend_norm columns: {missing}")

    X = stab_all[trend_cols].apply(pd.to_numeric, errors="coerce").to_numpy()
    X = np.abs(X)

    # --- normalize each temperature topic ---
    if robust:
        scale = np.nanmedian(X, axis=0)
    else:
        scale = np.nanstd(X, axis=0)

    scale = np.where(np.isfinite(scale) & (scale > 0), scale, np.nan)
    Xn = X / scale

    # --- RMS combine across topics ---
    thermal_activity = np.sqrt(np.nanmean(Xn**2, axis=1))
    n_used = np.sum(np.isfinite(Xn), axis=1)

    out = stab_all[group_cols].copy()
    out["thermal_activity"] = thermal_activity
    out["thermal_n_used"] = n_used

    # Keep individual trend_norms for diagnostics
    for c in trend_cols:
        out[c] = stab_all[c]

    return out


In [ ]:
TEMP_TOPICS = [
    "tma_truss_minus_xy",
    "mtcamera_ambAirtemp",
    "mtcamera_camBodyXPlusTemp",
    "mt_salindex113_temperature_0_mean",
]

thermal_activity = compute_composite_thermal_activity_from_raw(
    df=df,          # merged per-run stability table
    temp_cols=TEMP_TOPICS,
)


In [ ]:
merged = thermal_activity.merge(
    stab_aos_fwhm[["night", "test_index", "aos_fwhm_2_trend_tau"]],
    on=["night", "test_index"],
    how="left",
)

In [ ]:
for col in merged.columns:
    print(col)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap
from matplotlib.ticker import ScalarFormatter

def plot_spaghetti_centered_colored_by_run_metric(
    df: pd.DataFrame,
    value_col: str,
    run_metrics: pd.DataFrame,
    *,
    metric_col: str,
    group_cols=("night", "test_index"),
    time_rel_col="t_since_start_min",
    first_n: int = 5,
    max_tests: int | None = None,
    abs_metric: bool = False,
    cmap_name: str = "viridis",
    show_colorbar: bool = True,
    colorbar_label: str | None = None,
):
    """
    Spaghetti plot of value_col vs minutes since start, centered by subtracting
    mean(first_n). Each line colored by run-level metric_col.

    Parameters
    ----------
    df : DataFrame
        Raw time-series with group_cols and either time_rel_col OR enough info
        for add_test_relative_time(df).
    value_col : str
        Column to plot as spaghetti.
    run_metrics : DataFrame
        Per-run table containing group_cols + metric_col (one row per run).
        Example: stab_aos_fwhm[[night,test_index,aos_fwhm_2_trend_tau]]
                 or thermal_activity[[night,test_index,thermal_activity]]
    metric_col : str
        Column in run_metrics used for coloring.
    abs_metric : bool
        If True, color by abs(metric) (useful for tau).
    show_colorbar : bool
        If True, show a colorbar.
    """
    group_cols = list(group_cols)

    # Ensure relative time exists
    if time_rel_col not in df.columns:
        df_local = add_test_relative_time(df)
    else:
        df_local = df.copy()

    # numeric coercion
    df_local[value_col] = pd.to_numeric(df_local[value_col], errors="coerce")

    # merge in run metric
    rm = run_metrics[group_cols + [metric_col]].copy()
    rm[metric_col] = pd.to_numeric(rm[metric_col], errors="coerce")
    df_local = df_local.merge(rm, on=group_cols, how="left")

    # pick runs
    tests = (
        df_local[group_cols]
        .drop_duplicates()
        .sort_values(group_cols)
        .reset_index(drop=True)
    )
    if max_tests is not None:
        tests = tests.head(max_tests)

    if len(tests) == 0:
        print("No tests found.")
        return

    # color normalization
    mvals = rm[metric_col].to_numpy(dtype=float)
    mvals = mvals[np.isfinite(mvals)]
    if abs_metric:
        mvals = np.abs(mvals)

    if mvals.size == 0:
        vmin, vmax = 0.0, 1.0
    else:
        vmin, vmax = float(np.nanmin(mvals)), float(np.nanmax(mvals))
        if not np.isfinite(vmax) or vmax <= vmin:
            vmax = vmin + 1e-6

    norm = Normalize(vmin=vmin, vmax=vmax, clip=True)
    cmap = get_cmap(cmap_name)

    fig, ax = plt.subplots(figsize=(10, 6))
    n_plotted = 0

    for _, r in tests.iterrows():
        sel = (df_local[group_cols[0]] == r[group_cols[0]])
        for gc in group_cols[1:]:
            sel &= (df_local[gc] == r[gc])

        d = df_local.loc[sel, [time_rel_col, value_col, metric_col]].dropna(subset=[time_rel_col, value_col])
        if len(d) < 2:
            continue
        d = d.sort_values(time_rel_col)

        base = d[value_col].iloc[:first_n].mean()
        if not np.isfinite(base):
            continue

        y = d[value_col].to_numpy(dtype=float) - base
        x = d[time_rel_col].to_numpy(dtype=float)

        mv = pd.to_numeric(d[metric_col], errors="coerce").iloc[0]
        mv_c = abs(mv) if abs_metric and np.isfinite(mv) else mv

        color = cmap(norm(mv_c)) if np.isfinite(mv_c) else (0.6, 0.6, 0.6, 0.35)
        ax.plot(x, y, color=color, linewidth=1.2, alpha=0.75)

        n_plotted += 1

    ax.axhline(0.0, linestyle="--", linewidth=1.2)
    ax.set_xlabel("Minutes since test start")
    ax.set_ylabel(f"{value_col} − mean(first {first_n})")
    ax.set_title(f"{value_col}: centered spaghetti ({n_plotted} runs) colored by {metric_col}")
    ax.grid(True)

    # no scientific notation on y-axis
    fmt = ScalarFormatter(useMathText=False)
    fmt.set_scientific(False)
    fmt.set_useOffset(False)
    ax.yaxis.set_major_formatter(fmt)

    if show_colorbar:
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02)
        if colorbar_label is None:
            colorbar_label = f"|{metric_col}|" if abs_metric else metric_col
        cbar.set_label(colorbar_label)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_spaghetti_centered_colored_by_run_metric(
    df,
    value_col="aos_fwhm_2",
    run_metrics=thermal_activity[["night","test_index","thermal_activity"]],
    metric_col="thermal_activity",
    abs_metric=False,
    show_colorbar=True,
)

In [ ]:
plot_spaghetti_centered_colored_by_run_metric(
    df,
    value_col="aos_fwhm_2",
    run_metrics=thermal_activity[["night","test_index","thermal_activity"]],
    metric_col="thermal_activity",
    abs_metric=False,
    show_colorbar=True,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def compute_lagged_deltas(
    df: pd.DataFrame,
    value_cols: list[str],
    lag_minutes: float,
    *,
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
    tolerance=None,
) -> pd.DataFrame:
    """
    Compute time-lagged deltas within each run using merge_asof.

    For each row at time t, find the nearest earlier row at t - lag_minutes
    within the same group, then compute:
        delta_col = value(t) - value(t-lag)

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    value_cols : list[str]
        Columns to difference.
    lag_minutes : float
        Lag timescale in minutes.
    group_cols : tuple
        Grouping columns defining a run.
    time_col : str
        Time column.
    tolerance : str or pd.Timedelta or None
        Optional matching tolerance for merge_asof. If None, defaults to half the lag.

    Returns
    -------
    out : pd.DataFrame
        Copy of matched rows with lagged values and deltas.
    """
    lag = pd.Timedelta(minutes=lag_minutes)
    if tolerance is None:
        tolerance = lag / 2

    work = df.copy()
    work = work.dropna(subset=[time_col, *value_cols]).copy()
    work[time_col] = pd.to_datetime(work[time_col])

    # sort for merge_asof
    work = work.sort_values([*group_cols, time_col]).reset_index(drop=True)

    # left table: current values at time t
    left = work[[*group_cols, time_col, *value_cols]].copy()

    # right table: historical values shifted forward by lag,
    # so that a row from original time (t-lag) can match current time t
    right = work[[*group_cols, time_col, *value_cols]].copy()
    right[time_col] = right[time_col] + lag
    right = right.rename(columns={c: f"{c}_lag" for c in value_cols})

    out = pd.merge_asof(
        left.sort_values([*group_cols, time_col]),
        right.sort_values([*group_cols, time_col]),
        on=time_col,
        by=list(group_cols),
        direction="nearest",
        tolerance=tolerance,
    )

    for c in value_cols:
        out[f"delta_{c}"] = out[c] - out[f"{c}_lag"]

    out["lag_minutes"] = lag_minutes
    return out


def summarize_delta_relation(
    delta_df: pd.DataFrame,
    x_delta_col: str,
    y_delta_col: str,
) -> dict:
    """
    Summarize y vs x relationship for lagged deltas.
    """
    d = delta_df[[x_delta_col, y_delta_col]].dropna().copy()
    n = len(d)

    if n < 3:
        return {
            "n": n,
            "pearson_r": np.nan,
            "slope": np.nan,
            "intercept": np.nan,
        }

    x = d[x_delta_col].to_numpy()
    y = d[y_delta_col].to_numpy()

    pearson_r = np.corrcoef(x, y)[0, 1]

    # ordinary linear fit: y = m x + b
    slope, intercept = np.polyfit(x, y, 1)

    return {
        "n": n,
        "pearson_r": pearson_r,
        "slope": slope,
        "intercept": intercept,
    }


def plot_delta_relation(
    delta_df: pd.DataFrame,
    *,
    x_delta_col="delta_z_gradient",
    y_delta_col="delta_z4_1",
    title=None,
    alpha=0.35,
):
    """
    Scatter of delta z4 vs delta z-gradient with best-fit line.
    """
    d = delta_df[[x_delta_col, y_delta_col]].dropna().copy()

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(d[x_delta_col], d[y_delta_col], alpha=alpha)

    if len(d) >= 3:
        x = d[x_delta_col].to_numpy()
        y = d[y_delta_col].to_numpy()
        m, b = np.polyfit(x, y, 1)

        xx = np.linspace(np.nanmin(x), np.nanmax(x), 200)
        yy = m * xx + b
        ax.plot(xx, yy)

        r = np.corrcoef(x, y)[0, 1]
        ax.text(
            0.03, 0.97,
            f"n = {len(d)}\nr = {r:.3f}\nslope = {m:.3g}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

    ax.set_xlabel(r"$\Delta$ z_gradient")
    ax.set_ylabel(r"$\Delta$ z4 um")
    ax.set_title(title or f"{y_delta_col} vs {x_delta_col}")
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_delta_relation_panels(
    df: pd.DataFrame,
    lags=(5, 15, 30),
    *,
    z_col="z4_1",
    grad_col="z_gradient",
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
):
    """
    Make one panel per lag timescale.
    """
    fig, axes = plt.subplots(1, len(lags), figsize=(5 * len(lags), 4), constrained_layout=True)

    if len(lags) == 1:
        axes = [axes]

    for ax, lag in zip(axes, lags):
        dd = compute_lagged_deltas(
            df,
            value_cols=[z_col, grad_col],
            lag_minutes=lag,
            group_cols=group_cols,
            time_col=time_col,
        )

        d = dd[[f"delta_{grad_col}", f"delta_{z_col}"]].dropna()

        ax.scatter(d[f"delta_{grad_col}"], d[f"delta_{z_col}"], alpha=0.35)

        if len(d) >= 3:
            x = d[f"delta_{grad_col}"].to_numpy()
            y = d[f"delta_{z_col}"].to_numpy()
            m, b = np.polyfit(x, y, 1)
            r = np.corrcoef(x, y)[0, 1]

            xx = np.linspace(np.nanmin(x), np.nanmax(x), 200)
            ax.plot(xx, m * xx + b, 'r--')

            ax.text(
                0.03, 0.97,
                f"n = {len(d)}\nr = {r:.3f}\nslope = {m:.3g}",
                transform=ax.transAxes,
                va="top",
                ha="left",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
            )

        ax.axhline(0, linewidth=1)
        ax.axvline(0, linewidth=1)
        ax.set_xlabel(r"$\Delta$ z_gradient")
        ax.set_ylabel(r"$\Delta$ z4 um")
        ax.set_title(f"{lag} min")
        ax.grid(True, alpha=0.3)

    plt.show()

In [ ]:
plot_delta_relation_panels(
    df,
    lags=(5, 15, 30),
    z_col="z4_2",
    grad_col="z_gradient",
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
)

In [ ]:
import numpy as np
import pandas as pd

def compute_windowed_trend_deltas(
    df: pd.DataFrame,
    value_cols: list[str],
    window_minutes: float,
    *,
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
    min_points=5,
):
    """
    Estimate change over a window by fitting a line within each window.

    For each time t:
        - take data in [t - window, t]
        - fit x(t') = a + b t'
        - define delta_x = b * window

    Returns delta_x for each column.
    """
    window = pd.Timedelta(minutes=window_minutes)

    work = df.copy()
    work = work.dropna(subset=[time_col, *value_cols]).copy()
    work[time_col] = pd.to_datetime(work[time_col])
    work = work.sort_values([*group_cols, time_col]).reset_index(drop=True)

    results = []

    for group_key, g in work.groupby(list(group_cols), sort=False):
        g = g.sort_values(time_col).copy()

        t = g[time_col]
        t_sec = (t - t.iloc[0]).dt.total_seconds().to_numpy()

        out = g[[*group_cols, time_col]].copy()

        for col in value_cols:
            out[f"delta_{col}"] = np.nan

        for i in range(len(g)):
            t_now = t.iloc[i]
            mask = (t >= t_now - window) & (t <= t_now)

            sub = g.loc[mask]
            if len(sub) < min_points:
                continue

            tt = (sub[time_col] - sub[time_col].iloc[0]).dt.total_seconds().to_numpy()

            for col in value_cols:
                y = sub[col].to_numpy()

                good = np.isfinite(y)
                if good.sum() < min_points:
                    continue

                # linear fit
                slope, intercept = np.polyfit(tt[good], y[good], 1)

                # convert slope to change over window
                delta = slope * window.total_seconds()

                out.loc[g.index[i], f"delta_{col}"] = delta

        out["window_minutes"] = window_minutes
        results.append(out)

    if not results:
        return pd.DataFrame()

    return pd.concat(results, ignore_index=True)

def summarize_delta_relation(
    delta_df: pd.DataFrame,
    x_delta_col: str,
    y_delta_col: str,
) -> dict:
    """
    Summarize y vs x relationship for lagged deltas.
    """
    d = delta_df[[x_delta_col, y_delta_col]].dropna().copy()
    n = len(d)

    if n < 3:
        return {
            "n": n,
            "pearson_r": np.nan,
            "slope": np.nan,
            "intercept": np.nan,
        }

    x = d[x_delta_col].to_numpy()
    y = d[y_delta_col].to_numpy()

    pearson_r = np.corrcoef(x, y)[0, 1]

    # ordinary linear fit: y = m x + b
    slope, intercept = np.polyfit(x, y, 1)

    return {
        "n": n,
        "pearson_r": pearson_r,
        "slope": slope,
        "intercept": intercept,
    }


def plot_delta_relation(
    delta_df: pd.DataFrame,
    *,
    x_delta_col="delta_z_gradient",
    y_delta_col="delta_z4_1",
    title=None,
    alpha=0.35,
):
    """
    Scatter of delta z4 vs delta z-gradient with best-fit line.
    """
    d = delta_df[[x_delta_col, y_delta_col]].dropna().copy()

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(d[x_delta_col], d[y_delta_col], alpha=alpha)

    if len(d) >= 3:
        x = d[x_delta_col].to_numpy()
        y = d[y_delta_col].to_numpy()
        m, b = np.polyfit(x, y, 1)

        xx = np.linspace(np.nanmin(x), np.nanmax(x), 200)
        yy = m * xx + b
        ax.plot(xx, yy)

        r = np.corrcoef(x, y)[0, 1]
        ax.text(
            0.03, 0.97,
            f"n = {len(d)}\nr = {r:.3f}\nslope = {m:.3g}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

    ax.set_xlabel(r"$\Delta$ z_gradient")
    ax.set_ylabel(r"$\Delta$ z4 um")
    ax.set_title(title or f"{y_delta_col} vs {x_delta_col}")
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_delta_relation_panels(
    df: pd.DataFrame,
    lags=(5, 15, 30),
    *,
    z_col="z4_1",
    grad_col="z_gradient",
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
):
    """
    Make one panel per lag timescale.
    """
    fig, axes = plt.subplots(1, len(lags), figsize=(5 * len(lags), 4), constrained_layout=True)

    if len(lags) == 1:
        axes = [axes]

    for ax, lag in zip(axes, lags):
        dd = compute_windowed_trend_deltas(
            df,
            value_cols=[z_col, grad_col],
            window_minutes=lag,
            group_cols=group_cols,
            time_col=time_col,
        )

        d = dd[[f"delta_{grad_col}", f"delta_{z_col}"]].dropna()

        ax.scatter(d[f"delta_{grad_col}"], d[f"delta_{z_col}"], alpha=0.2, s=10)

        #hb = ax.hexbin(
        #    d[f"delta_{grad_col}"],
        #    d[f"delta_{z_col}"],
        #    gridsize=40,
        #    mincnt=1,
        #    cmap="viridis",
        #)
        
        #fig.colorbar(hb, ax=ax, label="counts")
        
        if len(d) >= 3:
            x = d[f"delta_{grad_col}"].to_numpy()
            y = d[f"delta_{z_col}"].to_numpy()
            m, b = np.polyfit(x, y, 1)
            r = np.corrcoef(x, y)[0, 1]

            xx = np.linspace(-0.11, 0.11, 20)
            ax.plot(xx, m * xx + b, 'r--')

            ax.text(
                0.03, 0.97,
                f"n = {len(d)}\nr = {r:.3f}\nslope = {m:.3g}",
                transform=ax.transAxes,
                va="top",
                ha="left",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
            )

        xticks = np.array([-0.05, -0.025, 0.0, 0.025, 0.05])
        ax.set_xticks(xticks)
        
        # define the ticks you want labeled
        label_values = np.array([-0.05, -0.025, 0.0, 0.025, 0.05])
        
        labels = []
        for t in xticks:
            if np.any(np.isclose(t, label_values, atol=1e-3)):
                labels.append(f"{t:+.2f}" if t != 0 else "0.0")
            else:
                labels.append("")
        ax.set_xticklabels(labels)

        ax.set_xlim(-0.05,0.05)
        ax.set_ylim(-1.1,1.1)
        ax.set_xlabel(r"$\Delta$ z_gradient")
        ax.set_ylabel(r"$\Delta$ z4 um")
        ax.set_title(f"{lag} min")
        ax.grid(True, alpha=0.3)

    plt.show()

In [ ]:
plot_delta_relation_panels(
    df,
    lags=(5, 10, 15),
    z_col="z4_2",
    grad_col="z_gradient",
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_delta_relation_panels_with_hist(
    df: pd.DataFrame,
    lags=(5, 15, 30),
    *,
    z_col="z4_1",
    grad_col="z_gradient",
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
    hist_bins=30,
    suptitle=None,
):
    """
    Two-row panel:
      Top: scatter (with fit)
      Bottom: histogram of delta_z_gradient with ±2σ lines

    Includes:
      - custom xticks and sparse labels
      - consistent limits
      - same point styling as original
    """

    ncols = len(lags)
    fig, axes = plt.subplots(
        2,
        ncols,
        figsize=(5 * ncols, 6),
        sharex="col",
        gridspec_kw={"height_ratios": [3, 1]},
        constrained_layout=True,
    )

    if ncols == 1:
        axes = np.array(axes).reshape(2, 1)

    # --- define ticks and labels once ---
    xticks = np.array([-0.05, -0.025, 0.0, 0.025, 0.05])
    label_values = np.array([-0.05, -0.025, 0.0, 0.025, 0.05])

    xtick_labels = []
    for t in xticks:
        if np.any(np.isclose(t, label_values, atol=1e-3)):
            xtick_labels.append(f"{t:+.2f}" if t != 0 else "0.0")
        else:
            xtick_labels.append("")

    for col, lag in enumerate(lags):
        ax = axes[0, col]
        ax_hist = axes[1, col]

        dd = compute_windowed_trend_deltas(
            df,
            value_cols=[z_col, grad_col],
            window_minutes=lag,
            group_cols=group_cols,
            time_col=time_col,
        )

        xcol = f"delta_{grad_col}"
        ycol = f"delta_{z_col}"
        d = dd[[xcol, ycol]].dropna()

        # --- TOP: scatter ---
        ax.scatter(d[xcol], d[ycol], alpha=0.2, s=10)

        if len(d) >= 3:
            x = d[xcol].to_numpy()
            y = d[ycol].to_numpy()
            m, b = np.polyfit(x, y, 1)
            r = np.corrcoef(x, y)[0, 1]

            xx = np.linspace(-0.11, 0.11, 20)
            ax.plot(xx, m * xx + b, "r--")

            ax.text(
                0.03,
                0.97,
                f"n = {len(d)}\nr = {r:.3f}\nslope = {m:.3g}",
                transform=ax.transAxes,
                va="top",
                ha="left",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
            )

        ax.set_xlim(-0.055, 0.055)
        ax.set_ylim(-0.05, 0.05)
        ax.set_ylabel(r"$\Delta$ z22 um")
        ax.set_title(f"{lag} min")
        ax.grid(True, alpha=0.3)

        # --- BOTTOM: histogram ---
        if len(d) > 0:
            x = d[xcol].to_numpy()
            mu = np.nanmean(x)
            sigma = np.nanstd(x)

            ax_hist.hist(x, bins=hist_bins, alpha=0.8)

        ax_hist.set_xlim(-0.055, 0.055)
        ax_hist.set_xlabel(r"$\Delta$ z_gradient")
        ax_hist.set_ylabel("N")
        ax_hist.grid(True, alpha=0.3)

        # --- apply custom ticks to BOTH rows ---
        ax.set_xticks(xticks)
        ax.set_xticklabels([])  # hide on top row

        ax_hist.set_xticks(xticks)
        ax_hist.set_xticklabels(xtick_labels)

    if suptitle is not None:
        fig.suptitle(suptitle)
        
    plt.show()

In [ ]:
plot_delta_relation_panels_with_hist(
    df,
    lags=(5, 10, 15),
    z_col="z22_2",
    grad_col="z_gradient",
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_delta_relation_panels_with_hist(
    df: pd.DataFrame,
    lags=(5, 10, 15),
    *,
    z_col="z4_1",
    grad_col="z_gradient",
    group_cols=("night", "test_index"),
    time_col="obs_start_dt",
    hist_bins=30,
    suptitle=None,
):
    ncols = len(lags)

    fig, axes = plt.subplots(
        2,
        ncols,
        figsize=(5 * ncols, 6),
        sharex="col",
        gridspec_kw={"height_ratios": [3, 1]},
        constrained_layout=True,
    )

    if ncols == 1:
        axes = np.array(axes).reshape(2, 1)

    # --- ticks + labels ---
    xticks = np.array([-0.1, -0.075, -0.05, -0.025, 0.0, 0.025, 0.05, 0.075, 0.1])
    label_values = np.array([-0.1, -0.05, 0.0, 0.05, 0.1])

    xtick_labels = [
        f"{t:+.2f}" if (np.any(np.isclose(t, label_values)) and t != 0)
        else ("0.0" if np.isclose(t, 0) else "")
        for t in xticks
    ]

    for col, lag in enumerate(lags):
        ax = axes[0, col]
        ax_hist = axes[1, col]

        dd = compute_windowed_trend_deltas(
            df,
            value_cols=[z_col, grad_col],
            window_minutes=lag,
            group_cols=group_cols,
            time_col=time_col,
        )

        xcol = f"delta_{grad_col}"
        ycol = f"delta_{z_col}"
        d = dd[[xcol, ycol]].dropna()

        # --- scatter ---
        ax.scatter(d[xcol], d[ycol], alpha=0.2, s=10)

        if len(d) >= 3:
            x = d[xcol].to_numpy()
            y = d[ycol].to_numpy()

            m, b = np.polyfit(x, y, 1)
            r = np.corrcoef(x, y)[0, 1]

            xx = np.linspace(-0.11, 0.11, 20)
            ax.plot(xx, m * xx + b, "r--")

            ax.text(
                0.03, 0.97,
                f"n = {len(d)}\nr = {r:.3f}\nslope = {m:.3g}",
                transform=ax.transAxes,
                va="top",
                ha="left",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
            )

        ax.axhline(0, linewidth=1)
        ax.axvline(0, linewidth=1)
        ax.set_xlim(-0.11, 0.11)
        ax.set_ylim(-1.1, 1.1)
        ax.set_ylabel(fr"$\Delta$ {z_col.replace('_2','')} $\mu$m")
        ax.set_title(f"{lag} min")
        ax.grid(True, alpha=0.3)

        # --- histogram ---
        if len(d) > 0:
            x = d[xcol].to_numpy()
            mu = np.nanmean(x)
            sigma = np.nanstd(x)

            ax_hist.hist(x, bins=hist_bins, alpha=0.8)
            ax_hist.axvline(mu - 2 * sigma, linestyle="--", linewidth=1.5)
            ax_hist.axvline(mu + 2 * sigma, linestyle="--", linewidth=1.5)
            ax_hist.axvline(mu, linestyle=":", linewidth=1.2)

        ax_hist.set_xlim(-0.11, 0.11)
        ax_hist.set_xlabel(r"$\Delta$ z_gradient")
        ax_hist.set_ylabel("N")
        ax_hist.grid(True, alpha=0.3)

        # --- ticks ---
        ax.set_xticks(xticks)
        ax.set_xticklabels([])

        ax_hist.set_xticks(xticks)
        ax_hist.set_xticklabels(xtick_labels)

    if suptitle:
        fig.suptitle(suptitle)

    return fig

In [ ]:
import os

output_dir = "zernike_delta_plots"
os.makedirs(output_dir, exist_ok=True)

for j in range(4, 29):
    z_col = f"z{j}_2"

    if z_col not in df.columns:
        print(f"Skipping {z_col}")
        continue

    print(f"Processing {z_col}...")

    fig = plot_delta_relation_panels_with_hist(
        df,
        lags=(5, 10, 15),
        z_col=z_col,
        grad_col="z_gradient",
        group_cols=("night", "test_index"),
        time_col="obs_start_dt",
        suptitle=f"{z_col} vs z_gradient",
    )

    filepath = os.path.join(output_dir, f"{z_col}_vs_z_gradient.png")
    fig.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)